# Carga de datos

**Objetivo:** importar la base crediticia y realizar controles iniciales antes del EDA.

Este notebook no modifica el archivo original. Solo lo carga en memoria y comprueba que esté listo para analizar.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Localizar y cargar el archivo

In [2]:
# Permite ejecutar el notebook desde la raíz del proyecto o desde src/.
posibles_rutas = [Path("Base_de_datos.csv"), Path("../Base_de_datos.csv")]
ruta_datos = next((ruta for ruta in posibles_rutas if ruta.exists()), None)

if ruta_datos is None:
    raise FileNotFoundError("No se encontró Base_de_datos.csv. Revisá la estructura del proyecto.")

df = pd.read_csv(ruta_datos, parse_dates=["fecha_prestamo"])
print(f"Archivo cargado desde: {ruta_datos.resolve()}")
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")

Archivo cargado desde: C:\Users\mchap\OneDrive\Escritorio\CURSOS\SOYHENRY\DATA SCIENCE\M5\mlops_pipeline\Base_de_datos.csv
Filas: 10,763 | Columnas: 23


## 2. Vista inicial

In [3]:
df.head()

,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,puntaje_datacredito,cant_creditosvigentes,huella_consulta,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,"3,692,160.00",10,42,Independiente,8000000,2500000,341296,88.77,695.00,10,5,0.00,"51,258.00","51,258.00",0.00,5,0,0,"908,526.00",Estable,1
1,4,2025-04-22 09:47:35,"840,000.00",6,60,Empleado,3000000,2000000,124876,95.23,789.00,3,1,0.00,"8,673.00","8,673.00",0.00,0,0,2,"939,017.00",Creciente,1
2,9,2026-01-08 12:22:40,"5,974,028.40",10,36,Independiente,4036000,829000,529554,47.61,740.00,4,5,0.00,"18,702.00","18,702.00",0.00,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,"1,671,240.00",6,48,Empleado,1524547,498000,252420,95.23,837.00,4,4,0.00,"15,782.00","15,782.00",0.00,3,0,0,"1,536,193.00",Creciente,1
4,9,2025-04-26 11:24:26,"2,781,636.00",11,44,Empleado,5000000,4000000,217037,95.23,771.00,4,6,0.00,"204,804.00","204,804.00",0.00,3,0,1,"933,473.00",Creciente,1


In [4]:
resumen_columnas = pd.DataFrame({
    "tipo_dato": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(2),
    "valores_unicos": df.nunique(dropna=True),
})
resumen_columnas

,tipo_dato,no_nulos,nulos,porcentaje_nulos,valores_unicos
tipo_credito,int64,10763,0,0.00,6
fecha_prestamo,datetime64[us],10763,0,0.00,10758
capital_prestado,float64,10763,0,0.00,7306
plazo_meses,int64,10763,0,0.00,18
edad_cliente,int64,10763,0,0.00,54
tipo_laboral,str,10763,0,0.00,2
salario_cliente,int64,10763,0,0.00,1385
total_otros_prestamos,int64,10763,0,0.00,1538
cuota_pactada,int64,10763,0,0.00,9836
puntaje,float64,10763,0,0.00,248


## 3. Controles básicos de calidad

In [5]:
print(f"Filas duplicadas exactas: {df.duplicated().sum()}")
print()
print("Distribución de la variable objetivo:")
display(df["Pago_atiempo"].value_counts(dropna=False).rename("cantidad").to_frame())
display((df["Pago_atiempo"].value_counts(normalize=True, dropna=False) * 100).round(2).rename("porcentaje").to_frame())

Filas duplicadas exactas: 0

Distribución de la variable objetivo:


,cantidad
Pago_atiempo,
1,10252
0,511


,porcentaje
Pago_atiempo,
1,95.25
0,4.75


In [6]:
valores_objetivo = set(df["Pago_atiempo"].dropna().unique())
assert valores_objetivo.issubset({0, 1}), f"Objetivo inesperado: {valores_objetivo}"
print("Control superado: Pago_atiempo es una variable binaria (0/1).")

Control superado: Pago_atiempo es una variable binaria (0/1).


## Conclusión de la carga

- La base se cargó correctamente.
- `Pago_atiempo` es la variable objetivo: `1` significa pago a tiempo y `0`, pago fuera de término.
- Los faltantes, valores atípicos y el desbalance de clases se estudian en `comprension_eda.ipynb`.